# Production Scheduling and Optimization (PSO) System

Run every cell in order. No manual configuration is required — this notebook installs
dependencies, generates realistic dummy data, builds the full MILP model, validates it,
solves it with HiGHS, exports results, and displays every chart inline.

See `docs/01_requirements.md` and `docs/02_mathematical_formulation.md` in the project
repository for the full business requirements and mathematical formulation this notebook
implements.

## 1. Install dependencies

In [ ]:
!pip install -q pyomo highspy matplotlib openpyxl pandas numpy
print('Dependencies installed.')

## 2. Get the project source

If you're running this notebook from inside the cloned/uploaded `pso_system/` project
folder, this cell just confirms the path. If running standalone in Colab, upload the
`pso_system` folder (or clone your repository containing it) so that `config.py`, `src/`,
etc. are importable.

In [ ]:
import sys, os

PROJECT_ROOT = os.getcwd()  # assumes the notebook is launched from the pso_system/ root,
                             # or that pso_system/ has been uploaded/cloned into the CWD
if os.path.basename(PROJECT_ROOT) != 'pso_system' and os.path.isdir(os.path.join(PROJECT_ROOT, 'pso_system')):
    PROJECT_ROOT = os.path.join(PROJECT_ROOT, 'pso_system')

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
assert os.path.exists('config.py'), 'config.py not found — upload/clone the pso_system project folder first.'

## 3. (Optional) Adjust configuration
Defaults match the documented Phase 5 dummy-data spec. Edit here if you want a different scale.

In [ ]:
from config import CONFIG

# Example overrides (uncomment to use):
# CONFIG.dummy_data.n_orders = 15
# CONFIG.dummy_data.n_machines = 4
CONFIG.solver.time_limit_seconds = 180
CONFIG.solver.tee = True
print('Dummy data config:', CONFIG.dummy_data)
print('Solver config:', CONFIG.solver)

## 4. Generate dummy data

In [ ]:
from src.generate_dummy_data import generate_all
tables = generate_all()
for name, df in tables.items():
    print(f'{name:28s} {len(df):4d} rows')

## 5. Build the full model (load → validate → sets → parameters → variables → constraints → objective)

In [ ]:
from src.model_builder import build_full_model
model, data, sets_obj, params = build_full_model()
print('Model built successfully.')

## 6. Solve

In [ ]:
from src.solver import solve_model
solve_result = solve_model(model)
print('Status:', solve_result.status)
print('Termination:', solve_result.termination_condition)
print('Objective (total cost):', solve_result.objective_value)
print('Solve time (s):', solve_result.solve_time_seconds)
if solve_result.infeasibility_report:
    print('\nINFEASIBILITY DIAGNOSTICS:')
    for line in solve_result.infeasibility_report:
        print(' -', line)

## 7. Extract results and export CSV / Excel reports

In [ ]:
from src.results import extract_results, export_results

assert solve_result.is_optimal_or_feasible, 'No usable solution found — see diagnostics above.'
results = extract_results(model, solve_result)
written = export_results(results)
print('Wrote', len(written), 'files to outputs/:')
for p in written:
    print(' -', p)

### KPI summary

In [ ]:
results.kpis

### Cost breakdown

In [ ]:
results.cost_breakdown

### Production schedule (first 20 rows)

In [ ]:
results.production_schedule.head(20)

## 8. Charts

In [ ]:
from src.visualization import (
    plot_machine_gantt, plot_production_timeline, plot_inventory_levels,
    plot_machine_utilization, plot_production_quantity, plot_order_completion,
    plot_cost_breakdown,
)
import matplotlib.pyplot as plt
from PIL import Image

def show(path):
    img = Image.open(path)
    plt.figure(figsize=(img.width / 100, img.height / 100))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

show(plot_machine_gantt(results))

In [ ]:
show(plot_production_timeline(results))

In [ ]:
show(plot_inventory_levels(results))

In [ ]:
show(plot_machine_utilization(results))

In [ ]:
show(plot_production_quantity(results))

In [ ]:
show(plot_order_completion(results))

In [ ]:
show(plot_cost_breakdown(results))

## 9. Download the combined Excel report (optional)
Run this cell in Colab to download `outputs/pso_report.xlsx` to your local machine.

In [ ]:
try:
    from google.colab import files
    files.download('outputs/pso_report.xlsx')
except ImportError:
    print('Not running in Google Colab — find the report at outputs/pso_report.xlsx')